# RND Weight Tolerance Simulation

## [1] 

Read the profile configuration from `configs/profile_p1.yaml` and use the values defined there.
This notebook now relies on the YAML file for `seconds_per_tick`, `num_ticks`, `weight_tolerance`, `transfer_wt`, `passenger_speed_kmh`, and `jeep_speed_kmh`.

In [10]:
import copy
import time
from pathlib import Path
import yaml
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

from utils_simplified import reuse_citygraph, reuse_ddm, generate_route_system
from utils.simulation import SimulationSetup
from utils.passenger import EDGE_SW, EDGE_EW, EDGE_RI

config_yaml_path = Path('configs/profile_p1.yaml')
with config_yaml_path.open('r', encoding='utf-8') as f:
    base_config = yaml.safe_load(f)

config = copy.deepcopy(base_config)

required_sim_keys = [
    'seconds_per_tick',
    'num_ticks',
    'weight_tolerance',
    'passenger_speed_kmh',
    'jeep_speed_kmh'
]
missing_sim_keys = [k for k in required_sim_keys if k not in config.get('simulation', {})]
if missing_sim_keys:
    raise KeyError(
        f"Missing required simulation config keys in {config_yaml_path}: {missing_sim_keys}"
    )

if 'travel_graph' not in config or 'transfer_wt' not in config['travel_graph']:
    raise KeyError(
        f"Missing required travel_graph config key 'transfer_wt' in {config_yaml_path}."
    )

walk_speed_kmh = float(config['simulation']['passenger_speed_kmh'])
ride_speed_kmh = float(config['simulation']['jeep_speed_kmh'])

print(f"Using seconds_per_tick={config['simulation']['seconds_per_tick']}")
print(f"Using num_ticks={config['simulation']['num_ticks']}")
print(f"Using transfer_wt={config['travel_graph']['transfer_wt']} and weight_tolerance={config['simulation']['weight_tolerance']}")
print(f"walk_speed_kmh={walk_speed_kmh}, ride_speed_kmh={ride_speed_kmh}")


Using seconds_per_tick=30
Using num_ticks=7200
Using transfer_wt=15.78 and weight_tolerance=100
walk_speed_kmh=4.5, ride_speed_kmh=20.0


## [2]

Construct the city graph and demand sampler from the profile, and define the expected travel time computation using consistent km/h units.
The expected travel time is the sum of walk time and ride time, where edge lengths are converted from meters to kilometers before dividing by speed.

In [11]:
output_dir = Path('outputs/rnd_weight_tolerance')
output_dir.mkdir(parents=True, exist_ok=True)

cg_pkl = 'rnd/pkl/profile_p1.pkl'
ddm_pkl = 'rnd/pkl/ddm_8am.pkl'

print('Loading cached CityGraph and DirectDemandSampler...')
t0 = time.time()
city_graph = reuse_citygraph(cg_pkl)
ddm = reuse_ddm(ddm_pkl)
print(f"Loaded in {time.time() - t0:.1f}s | {len(city_graph.nodes)} nodes, {len(city_graph.graph)} edges")

EDGE_TYPE_LABELS = {EDGE_SW: 'SW', EDGE_EW: 'EW', EDGE_RI: 'RI'}

def compute_expected_travel_minutes(journey):
    walk_meters = sum(e.getLength() for e in journey if e._edge_type in (EDGE_SW, EDGE_EW))
    ride_meters = sum(e.getLength() for e in journey if e._edge_type == EDGE_RI)
    walk_minutes = walk_meters / 1000.0 / walk_speed_kmh * 60.0
    ride_minutes = ride_meters / 1000.0 / ride_speed_kmh * 60.0
    return walk_minutes, ride_minutes, walk_minutes + ride_minutes

print('CityGraph and DDM are ready.')


Loading cached CityGraph and DirectDemandSampler...
[INFO] Reusing CityGraph from pickle file: rnd/pkl/profile_p1.pkl
[INFO] Reusing DirectDemandSampler from pickle file: rnd/pkl/ddm_8am.pkl
Loaded in 5.7s | 36866 nodes, 76310 edges
CityGraph and DDM are ready.


## [3]

Execute each route count scenario and collect: planned EIVM journey details, actual traveled path edges, expected walk and ride times, and actual travel time when passengers complete.

In [12]:
from utils.passenger import Passenger

rows = []
summary_rows = []
scenarios = [12, 24, 36, 48, 60]  # Example scenarios with varying number of routes

for num_routes in scenarios:
    print(f'--- Scenario: {num_routes} routes ---')
    routes = generate_route_system(num_routes, city_graph, ddm)
    sim_setup = SimulationSetup(city_query=config['city_graph']['name'], config=config, routes=routes)
    sim = sim_setup.build()
    if not hasattr(Passenger, 'expected_route_idx'):
        Passenger.expected_route_idx = None

    result = sim.run()

    completed = []
    for p in sim.passenger_generator.archived_passengers:
        walk_min, ride_min, expected_min = compute_expected_travel_minutes(p.journey)
        actual_min = (p.despawn_tick - p.spawn_tick) / 60.0 if p.despawn_tick is not None else None
        rows.append({
            'num_routes': num_routes,
            'passenger_id': p.id,
            'completed': True,
            'spawn_tick': p.spawn_tick,
            'despawn_tick': p.despawn_tick,
            'actual_travel_time_min': actual_min,
            'expected_walk_time_min': walk_min,
            'expected_ride_time_min': ride_min,
            'expected_total_time_min': expected_min,
            'planned_cost_eivm': p.total_path_cost,
            'edge_sequence': ','.join(e.id for e in p.journey),
            'edge_types': ','.join(EDGE_TYPE_LABELS.get(e._edge_type, str(e._edge_type)) for e in p.journey),
            'num_edges': len(p.journey),
            'sim_ticks': sim.current_tick,
            'scenario_fitness': result.fitness_score,
            'boarded_expected': getattr(p, 'boarded_expected', None),
            'took_alternative': getattr(p, 'took_alternative', None),
        })
        completed.append((expected_min, actual_min))

    for p in sim.passenger_generator.passengers:
        walk_min, ride_min, expected_min = compute_expected_travel_minutes(p.journey)
        rows.append({
            'num_routes': num_routes,
            'passenger_id': p.id,
            'completed': False,
            'spawn_tick': p.spawn_tick,
            'despawn_tick': None,
            'actual_travel_time_min': None,
            'expected_walk_time_min': walk_min,
            'expected_ride_time_min': ride_min,
            'expected_total_time_min': expected_min,
            'planned_cost_eivm': p.total_path_cost,
            'edge_sequence': ','.join(e.id for e in p.journey),
            'edge_types': ','.join(EDGE_TYPE_LABELS.get(e._edge_type, str(e._edge_type)) for e in p.journey),
            'num_edges': len(p.journey),
            'sim_ticks': sim.current_tick,
            'scenario_fitness': result.fitness_score,
            'boarded_expected': getattr(p, 'boarded_expected', None),
            'took_alternative': getattr(p, 'took_alternative', None),
        })

    completed_expected = [x for x, _ in completed]
    completed_actual = [y for _, y in completed]
    summary_rows.append({
        'num_routes': num_routes,
        'completed_passengers': len(completed_actual),
        'incomplete_passengers': len(sim.passenger_generator.passengers),
        'sim_ticks': sim.current_tick,
        'scenario_fitness': result.fitness_score,
        'expected_vs_actual_count': len(completed_actual),
        'mean_expected_time_min': np.mean(completed_expected) if completed_expected else None,
        'mean_actual_time_min': np.mean(completed_actual) if completed_actual else None,
    })

    print(f'Scenario {num_routes} finished: {len(completed_actual)} completed, {len(sim.passenger_generator.passengers)} incomplete')


--- Scenario: 12 routes ---
[INFO] Generating 12 routes...
[Setup] Initializing CityGraph...
[Setup] Initializing Direct Demand Sampler...
[Setup] Injecting Transit Routes into Travel Graph...
[Setup] Deploying Fleet...
[Setup] Initializing Passenger Spawner...


KeyboardInterrupt: 

## [4]

Write the passenger-level journey and scenario summary tables to CSV. Also verify that the expected total travel time equals the sum of expected walk and ride time.

In [ ]:
df = pd.DataFrame(rows)
summary_df = pd.DataFrame(summary_rows)
df['expected_walk_time_min'] = df['expected_walk_time_min'].astype(float)
df['expected_ride_time_min'] = df['expected_ride_time_min'].astype(float)
df['expected_total_time_min'] = df['expected_total_time_min'].astype(float)
df['expected_total_time_check_min'] = df['expected_walk_time_min'] + df['expected_ride_time_min']
assert np.allclose(df['expected_total_time_min'], df['expected_total_time_check_min'], atol=1e-6), 'Expected total time must equal walk + ride time'

passenger_csv = output_dir / '2_weight_tolerance_passenger_results.csv'
summary_csv = output_dir / '2_weight_tolerance_summary.csv'
df.to_csv(passenger_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

sigma_expected_walk = df['expected_walk_time_min'].sum()
sigma_expected_ride = df['expected_ride_time_min'].sum()
sigma_expected_total = df['expected_total_time_min'].sum()
print(f'Σ expected_walk_time_min = {sigma_expected_walk:.4f} minutes')
print(f'Σ expected_ride_time_min = {sigma_expected_ride:.4f} minutes')
print(f'Σ expected_total_time_min = {sigma_expected_total:.4f} minutes')
print('Verified expected total time equals walk + ride time for all rows')
print(f'Passenger CSV exported to: {passenger_csv}')
print(f'Summary CSV exported to: {summary_csv}')

alt_taken = df['took_alternative'].eq(True).sum()
total_rows = len(df)
alt_pct = alt_taken / total_rows * 100 if total_rows else 0.0
missing_took_alternative = df['took_alternative'].isna().sum()
print(f'{alt_taken} passengers took an alternative route out of {total_rows} rows ({alt_pct:.2f}%).')
if missing_took_alternative:
    print(f'{missing_took_alternative} rows have missing took_alternative values.')

df.head(10)


Σ expected_walk_time_min = 58298.7877 minutes
Σ expected_ride_time_min = 121615.2743 minutes
Σ expected_total_time_min = 179914.0620 minutes
Verified expected total time equals walk + ride time for all rows
Passenger CSV exported to: outputs/rnd_weight_tolerance/2_weight_tolerance_passenger_results.csv
Summary CSV exported to: outputs/rnd_weight_tolerance/2_weight_tolerance_summary.csv
0 passengers took an alternative route out of 2824 rows (0.00%).


,num_routes,passenger_id,completed,spawn_tick,despawn_tick,actual_travel_time_min,expected_walk_time_min,expected_ride_time_min,expected_total_time_min,planned_cost_eivm,edge_sequence,edge_types,num_edges,sim_ticks,scenario_fitness,boarded_expected,took_alternative,expected_total_time_check_min
0,12,P0544ebb2555f4bb883837b8d60832d74,True,1260,1890.0,10.500000,0.000000,8.323632,8.323632,31.975119,"WA10310,RI_R6_21850,RI_R6_21851,RI_R6_21852,RI...","2,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI...",86,1000,2.352979e+06,True,False,8.323632
1,12,P6968ccaecf2b4fd5a063e30f1338def7,True,1750,2480.0,12.166667,0.165307,9.629013,9.794320,35.423129,"SW04212,WA08046,RI_R5_17283,RI_R5_17284,RI_R5_...","SW,2,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI...",104,1000,2.352979e+06,True,False,9.794320
2,12,Pb825bcdedd0c4bd8abc13a2945d55a59,True,1140,2650.0,25.166667,16.850625,4.365572,21.216197,110.568570,"SW75020,SW75018,SW75016,SW75014,SW75012,SW7501...","SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,S...",183,1000,2.352979e+06,True,False,21.216197
3,12,Pc51af1f7dba544a7a198b4cf91ec5e31,True,1010,2690.0,28.000000,6.173820,14.466929,20.640749,70.985953,"SW09448,SW09446,SW09444,SW09442,SW09440,SW0943...","SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,S...",201,1000,2.352979e+06,True,False,20.640749
4,12,P9c5e298e2ae043d78f9a3229708dc60f,True,570,2820.0,37.500000,15.423631,18.460818,33.884449,134.237073,"SW28171,SW28173,SW28175,SW28177,SW28179,SW2818...","SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,S...",301,1000,2.352979e+06,True,False,33.884449
5,12,Pf984b5f3576c46e99c01b11d39bc9858,True,570,3160.0,43.166667,36.847577,5.197813,42.045390,180.978952,"SW15070,SW15068,SW15066,SW07808,SW07805,SW0780...","SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,S...",197,1000,2.352979e+06,True,False,42.045390
6,12,Pc92b696e6ce84467b35a9ab4042fc7e1,True,210,3240.0,50.500000,24.223392,23.978416,48.201808,167.237802,"WA08309,RI_R5_17546,RI_R5_17547,RI_R5_17548,RI...","2,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI,RI...",674,1000,2.352979e+06,True,False,48.201808
7,12,P0c4c4aef2d5043029738418a16e3536a,True,1240,3260.0,33.666667,12.063191,15.936489,27.999680,114.729694,"SW13758,SW13752,SW13755,SW13800,SW13798,SW1379...","SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,S...",197,1000,2.352979e+06,True,False,27.999680
8,12,P548db46849bf42c6aa94152d5f665531,True,560,3280.0,45.333333,3.063345,37.433392,40.496736,122.014652,"SW36022,SW36020,SW36018,SW36016,SW36014,SW3601...","SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,S...",657,1000,2.352979e+06,True,False,40.496736
9,12,Peac8ffc2327f4b9a9e19b1b5d23d4b18,True,120,3290.0,52.833333,3.441583,33.889494,37.331077,100.365952,"SW65048,SW65045,SW65051,SW65053,SW65057,SW6505...","SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,SW,S...",480,1000,2.352979e+06,True,False,37.331077
